# Hotel Occupancy & Room-Demand Forecasting Demo
End-to-end walkthrough: Data Quality Checks, EDA, Chronological Validation, Model Evaluation, and Error Inspection.

In [ ]:
import sys
sys.path.append('..')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
from src.features import FEATURE_COLS, build_features
from src.train import evaluate_predictions

## 1. Load Data & Check Quality

In [ ]:
df = pd.read_csv('../data/hotel_occupancy.csv')
df['stay_date'] = pd.to_datetime(df['stay_date'])
df['occupancy_rate'] = (df['occupied_rooms'] / df['room_capacity']) * 100

print('Total records:', len(df))
print('Missing values:', df.isnull().sum().sum())
print('Capacity violations:', (df['occupied_rooms'] > df['room_capacity']).sum())
df.head()

## 2. Exploratory Data Analysis (Seasonality & Weekly Pattern)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df.groupby('room_category')['occupancy_rate'].mean().plot(kind='bar', ax=axes, color='#2b5c8f')
axes.set_title('Mean Occupancy (%) by Room Category')
axes.set_ylabel('Occupancy (%)')
axes.tick_params(axis='x', rotation=0)

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['day_name'] = df['stay_date'].dt.day_name()
df.groupby('day_name')['occupancy_rate'].mean().reindex(day_order).plot(kind='line', marker='o', ax=axes, color='#d95f02')
axes.set_title('Weekly Occupancy Pattern')
axes.set_ylabel('Occupancy (%)')
axes.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 3. Chronological Split & Held-Out Test Evaluation

In [ ]:
feat_df = build_features(df).dropna().reset_index(drop=True)

train_df = feat_df[feat_df['stay_date'] < '2025-01-01']
val_df = feat_df[(feat_df['stay_date'] >= '2025-01-01') & (feat_df['stay_date'] < '2025-07-01')]
test_df = feat_df[feat_df['stay_date'] >= '2025-07-01'].copy()

print(f'Train rows: {len(train_df)} | Val rows: {len(val_df)} | Test rows: {len(test_df)}')

artifact = joblib.load('../models/demand_forecaster.joblib')
model = artifact['point_model']

test_df['pred_model'] = np.clip(model.predict(test_df[FEATURE_COLS]), 0, test_df['room_capacity'])
test_df['pred_lower'] = np.clip(artifact['lower_model'].predict(test_df[FEATURE_COLS]), 0, test_df['pred_model'])
test_df['pred_upper'] = np.clip(artifact['upper_model'].predict(test_df[FEATURE_COLS]), test_df['pred_model'], test_df['room_capacity'])

print('Naive Lag-7 Test Metrics:', evaluate_predictions(test_df['occupied_rooms'], test_df['occ_lag_7d']))
print('Trained Model Test Metrics:', evaluate_predictions(test_df['occupied_rooms'], test_df['pred_model']))

## 4. Forecast vs. Actuals & Most Influential Inputs

In [ ]:
sample_cat = test_df[(test_df['room_category'] == 'Deluxe') & (test_df['stay_date'] >= '2025-11-15')]

plt.figure(figsize=(11, 4))
plt.plot(sample_cat['stay_date'], sample_cat['occupied_rooms'], label='Actual Occupied', marker='o', color='black')
plt.plot(sample_cat['stay_date'], sample_cat['pred_model'], label='Model Forecast', marker='x', color='#1f78b4')
plt.fill_between(sample_cat['stay_date'], sample_cat['pred_lower'], sample_cat['pred_upper'], color='#a6cee3', alpha=0.4, label='80% Interval')
plt.title('Deluxe Room Forecast vs Actuals (Nov-Dec 2025 Held-Out Test)')
plt.ylabel('Room Nights')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

perm = permutation_importance(model, test_df[FEATURE_COLS], test_df['occupied_rooms'], n_repeats=5, random_state=42)
imp = pd.Series(perm.importances_mean, index=FEATURE_COLS).sort_values(ascending=True).tail(10)

plt.figure(figsize=(8, 4))
imp.plot(kind='barh', color='#33a02c')
plt.title('Top 10 Most Influential Model Inputs (Permutation Importance)')
plt.xlabel('Mean MAE/R2 Impact')
plt.tight_layout()
plt.show()

## 5. Single Prediction Error Inspection

In [ ]:
test_df['abs_error'] = (test_df['occupied_rooms'] - test_df['pred_model']).abs()
worst_case = test_df.sort_values('abs_error', ascending=False).iloc
print('Largest Single-Day Error Example:')
print(worst_case[['stay_date', 'room_category', 'room_capacity', 'is_holiday', 'otb_7d_prior', 'occupied_rooms', 'pred_model', 'abs_error']])